In [1]:
!pip install -q -U langgraph langchain-core openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 53.6 MB/s eta 0:00:00


In [6]:
# ============================================================
#        LANGGRAPH AGENT USING OPENAI API
#        SIMPLE AI RESPONSE VALIDATION PROJECT
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import os

# Used for type definition of graph state
from typing import TypedDict

# Import LangGraph components
from langgraph.graph import StateGraph, START, END

# Import OpenAI client
from openai import OpenAI

# Import Google Colab userdata
from google.colab import userdata


# ============================================================
# 2. GET OPENAI API KEY
# ============================================================

# Get OpenAI API key from Google Colab Secrets
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")


# Check whether API key was found
if not OPENAI_API_KEY:
    raise ValueError(
        "OPENAI_API_KEY not found. "
        "Please add it to Google Colab Secrets."
    )


print("OpenAI API key loaded successfully.")


# ============================================================
# 3. CREATE OPENAI CLIENT
# ============================================================

client = OpenAI(
    api_key=OPENAI_API_KEY
)


print("OpenAI client created successfully.")


# ============================================================
# 4. SELECT OPENAI MODEL
# ============================================================

MODEL_NAME = "gpt-5.6-luna"

print(
    "Model selected:",
    MODEL_NAME
)


# ============================================================
# 5. DEFINE THE GRAPH STATE SCHEMA
# ============================================================

class AgentState(TypedDict):

    # Stores the question given by the user
    user_query: str

    # Stores the response generated by the LLM
    response: str

    # Stores whether the response passed validation
    is_valid: bool


# ============================================================
# 6. GENERATOR NODE
# ============================================================

def generate_response_node(
    state: AgentState
):

    # Get user's question
    query = state["user_query"]


    # Create prompt for the LLM
    prompt = f"""
You are a helpful AI assistant.

Answer the following user question
clearly and in a simple way.

User Question:
{query}

Give a useful and meaningful answer.
"""


    # Send request to OpenAI
    response = client.responses.create(

        # Select OpenAI model
        model=MODEL_NAME,

        # Send prompt
        input=prompt
    )


    # Extract generated text
    generated_answer = response.output_text


    # Return updated state
    return {
        "response": generated_answer
    }


# ============================================================
# 7. VALIDATION NODE
# ============================================================

def validation_node(
    state: AgentState
):

    # Get generated response
    response = state["response"]


    # Check whether response contains
    # more than 10 characters
    valid = len(response.strip()) > 10


    # Return validation result
    return {
        "is_valid": valid
    }


# ============================================================
# 8. DEFINE ROUTING LOGIC
# ============================================================

def router(
    state: AgentState
):

    # Check validation result
    if state["is_valid"]:

        # Response is valid
        return "approved"

    else:

        # Response is not valid
        return "rejected"


# ============================================================
# 9. BUILD LANGGRAPH STATE MACHINE
# ============================================================

# Create StateGraph using AgentState
builder = StateGraph(AgentState)


# Add Generator Node
builder.add_node(
    "generator",
    generate_response_node
)


# Add Validator Node
builder.add_node(
    "validator",
    validation_node
)


# ============================================================
# 10. DEFINE GRAPH EDGES
# ============================================================

# Start with generator
builder.add_edge(
    START,
    "generator"
)


# Generator → Validator
builder.add_edge(
    "generator",
    "validator"
)


# ============================================================
# 11. CONDITIONAL ROUTING
# ============================================================

builder.add_conditional_edges(

    # Routing starts from validator
    "validator",

    # Routing function
    router,

    # Possible routes
    {
        # If approved → finish
        "approved": END,

        # If rejected → generate again
        "rejected": "generator"
    }
)


# ============================================================
# 12. COMPILE LANGGRAPH APPLICATION
# ============================================================

app = builder.compile()


print(
    "LangGraph compiled successfully!"
)


# ============================================================
# 13. EXECUTE GRAPH
# ============================================================

# Ask user for a question
user_question = input(
    "Enter your question: "
)


# Execute LangGraph
output = app.invoke({

    "user_query": user_question,

    "response": "",

    "is_valid": False
})


# ============================================================
# 14. DISPLAY RESULT
# ============================================================

print(
    "\n--- LANGGRAPH EXECUTION COMPLETE ---"
)


print(
    "\nFinal State:"
)


print(output)


print(
    "\nAI Response:"
)


print(
    output["response"]
)

OpenAI API key loaded successfully.
OpenAI client created successfully.
Model selected: gpt-5.6-luna
LangGraph compiled successfully!
Enter your question: What is the Capital of Russia

--- LANGGRAPH EXECUTION COMPLETE ---

Final State:
{'user_query': 'What is the Capital of Russia', 'response': 'The capital of Russia is **Moscow**.', 'is_valid': True}

AI Response:
The capital of Russia is **Moscow**.
